# 🔧 Port 5066 Debug Notebook
Run each cell in order. Each cell is a self-contained test.  
**Goal:** confirm port 5066 works before using `start_monitor`.

## Cell 1 — Configuration

In [1]:
# Edit these to match your setup
RP_IP    = "192.168.0.99"
RP_KEY   = "Cav"
SSH_USER = "root"
SSH_PASS = "root"
print("Config OK")

Config OK


## Cell 2 — Verify RP_Lock.py on board has port 5066

In [2]:
import paramiko

ssh = paramiko.SSHClient()
ssh.set_missing_host_key_policy(paramiko.AutoAddPolicy())
ssh.connect(RP_IP, port=22, username=SSH_USER, password=SSH_PASS, timeout=5)

_, out, _ = ssh.exec_command("grep -n '5066\|_mon_stop\|ch_byte\[0\]' /root/RP_Lock.py")
lines = out.read().decode().strip()
print("Matching lines in /root/RP_Lock.py:")
print(lines if lines else "  NONE — wrong file on board!")

_, out, _ = ssh.exec_command("ps aux | grep RunLock | grep -v grep")
proc = out.read().decode().strip()
print("\nRunLock.py process:", proc if proc else "NOT RUNNING")

ssh.close()

Matching lines in /root/RP_Lock.py:
395:            # Start port 5066 monitor server so the PC-side Monitor can keep
398:            _lock_mon_stop = _thr.Event()
403:                srv.bind(("", 5066))   # all interfaces
406:                print("Lock monitor server listening on port 5066")
407:                while not _lock_mon_stop.is_set():
415:                        ch = ch_byte[0] if ch_byte else 0
433:            _lock_mon_stop.clear()
435:            print("Lock monitor server started on port 5066")
438:            _lock_mon_stop.set()  # stop monitor server
506:            Dedicated monitor server on port 5066.
514:            srv.bind(("", 5066))   # all interfaces
517:            print("Monitor server listening on port 5066")
518:            while not _mon_stop.is_set():
526:                    ch = ch_byte[0] if ch_byte else 0
542:            print("Monitor server on port 5066 stopped")
544:        _mon_stop = _thr.Event()   # define the stop-event for _monitor_server
5

## Cell 3 — Force-upload correct RP_Lock.py to board

In [3]:

import paramiko, sys
from pathlib import Path

# Find RP_Lock.py in project
rp_lock_path = None
for p in sys.path:
    f = Path(p) / "RP_side" / "RP_Lock.py"
    if f.exists():
        rp_lock_path = f
        break

if rp_lock_path is None:
    print("ERROR: Could not find RP_side/RP_Lock.py in sys.path")
    print("Manually set: rp_lock_path = Path(r'C:/your/path/RP_side/RP_Lock.py')")
else:
    content = rp_lock_path.read_text(encoding="utf-8")
    has_5066    = "5066" in content
    has_ch_fix  = "ch_byte[0]" in content
    has_stop    = "_mon_stop" in content
    
    print(f"Local file: {rp_lock_path}")
    print(f"  Has port 5066  : {has_5066}")
    print(f"  Has ch_byte[0] : {has_ch_fix}  ← MUST BE True")
    print(f"  Has _mon_stop  : {has_stop}   ← MUST BE True")
    
    if not (has_5066 and has_ch_fix and has_stop):
        print("\nLocal file is OLD. Upload the fixed RP_Lock.py from outputs first!")
    else:
        # Kill old process and upload new file
        ssh = paramiko.SSHClient()
        ssh.set_missing_host_key_policy(paramiko.AutoAddPolicy())
        ssh.connect(RP_IP, port=22, username=SSH_USER, password=SSH_PASS)
        ssh.exec_command("pkill -f RunLock.py")
        import time; time.sleep(1)
        sftp = ssh.open_sftp()
        sftp.open("/root/RP_Lock.py", "w").write(content)
        sftp.close()
        _, out, _ = ssh.exec_command("grep -c '5066' /root/RP_Lock.py")
        print(f"\nUploaded. Lines with 5066 on board: {out.read().decode().strip()}")
        ssh.close()
        print("Ready — run Cell 4 to reconnect")

Local file: RP_side\RP_Lock.py
  Has port 5066  : True
  Has ch_byte[0] : True  ← MUST BE True
  Has _mon_stop  : True   ← MUST BE True

Uploaded. Lines with 5066 on board: 9
Ready — run Cell 4 to reconnect


## Cell 4 — Reconnect LockClient (uploads new RP_Lock.py automatically)

In [4]:
import sys, pathlib, threading, time

_here = pathlib.Path().resolve()
for _p in [_here] + list(_here.parents)[:4]:
    if (_p / "lockclient.py").exists():
        if str(_p) not in sys.path:
            sys.path.insert(0, str(_p))
        print("Repo root:", _p)
        break

from lockclient import LockClient, RP_client, Monitor

RPs = {
    "Cav": RP_client((RP_IP, 5000), {}, mode="scan_mon"),
}

print("Uploading scripts and loading settings...")
Lock = LockClient(RPs)
print("Done")

def _wrap(fn, err):
    try: fn()
    except Exception as exc: err["exc"] = exc

err = {}
t = threading.Thread(target=lambda: _wrap(Lock.connect_all, err), daemon=True)
t.start(); t.join(timeout=45)
if t.is_alive(): print("TIMEOUT connecting")
elif "exc" in err: print("ERROR:", err["exc"])
else: print("Connected OK")

stcl_thread = threading.Thread(target=Lock.start, daemon=True)
stcl_thread.start()
time.sleep(2)

from lockclient import Monitor
SHOW_TRIGGER = True
Monitor.show_trigger = SHOW_TRIGGER
print("Event loop started")

Repo root: C:\Users\RikteemBhowmick\Projects 2025\RedPitaya Projects\RP-STCL
Uploading scripts and loading settings...
Done
connecting...
Connected OK
Event loop started


## Cell 5 — Start scan loop

In [5]:
CAV_DEC    = 32
CAV_AMP    = 0.7
CAV_OFFSET = 0.0
_CAV_PERIOD_MS = 8e-9 * 16384 * CAV_DEC * 1e3

Lock.set_dec("Cav", CAV_DEC)
Lock.start_scan("Cav", amplitude=CAV_AMP, offset=CAV_OFFSET)
import time; time.sleep(3)   # wait for scan + port 5066 thread to start
print("Scan started. loop_running:", Lock.RPs["Cav"].loop_running)

connected to <socket.socket fd=2536, family=2, type=1, proto=0, laddr=('192.168.0.147', 59432), raddr=('192.168.0.99', 5065)>
Scan started. loop_running: True


## Cell 6 — Check which ports are listening on board

In [6]:
import paramiko

ssh = paramiko.SSHClient()
ssh.set_missing_host_key_policy(paramiko.AutoAddPolicy())
ssh.connect(RP_IP, port=22, username=SSH_USER, password=SSH_PASS)
_, out, _ = ssh.exec_command("ss -tlnp | grep -E '5000|5065|5066'")
result = out.read().decode().strip()
ssh.close()

print("Open ports on board:")
for line in result.splitlines():
    print(" ", line)

if "5066" in result:
    print("\n✓ Port 5066 is OPEN — proceed to Cell 7")
else:
    print("\n✗ Port 5066 NOT open")
    print("  Possible causes:")
    print("  1. Wrong RP_Lock.py on board (re-run Cell 3)")
    print("  2. _monitor_server thread crashed — check board console")

Open ports on board:
  LISTEN 0      128     192.168.0.99:5000      0.0.0.0:*    users:(("python3",pid=1496,fd=7))                      
  LISTEN 0      128     192.168.0.99:5065      0.0.0.0:*    users:(("python3",pid=1496,fd=12))                     
  LISTEN 0      5            0.0.0.0:5066      0.0.0.0:*    users:(("python3",pid=1496,fd=10))

✓ Port 5066 is OPEN — proceed to Cell 7


## Cell 7 — Raw test of port 5066 (verbose)

In [7]:
import socket, struct, json

print(f"Connecting to {RP_IP}:5066...")
s = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
s.settimeout(5)

try:
    s.connect((RP_IP, 5066))
    print("  Connected OK")
    
    print("  Sending channel byte 0...")
    s.sendall(bytes([0]))
    
    print("  Reading response...")
    s.settimeout(10)
    raw = b""
    length = None
    while True:
        chunk = s.recv(65536)
        if not chunk:
            print(f"  Board closed connection after {len(raw)} bytes")
            break
        raw += chunk
        print(f"  Got {len(chunk)} bytes (total {len(raw)})")
        if length is None and len(raw) >= 4:
            length = struct.unpack(">I", raw[:4])[0]
            print(f"  Payload length from header: {length} bytes")
        if length and len(raw) >= 4 + length:
            print("  Full response received!")
            break

    s.close()
    
    if len(raw) == 0:
        print("\n✗ ZERO BYTES received")
        print("  Board connected but sent nothing")
        print("  Fix: upload RP_Lock.py with ch_byte[0] fix (Cell 3)")
    elif length and len(raw) >= 4 + length:
        payload = raw[4:4+length]
        dur, data = json.loads(payload.decode("utf-8"))
        print(f"\n✓ SUCCESS!")
        print(f"  Duration : {dur:.4f} ms")
        print(f"  Samples  : {len(data)}")
        print(f"  Sample   : {data[:3]}")
        print("\nPort 5066 works — run Cell 8 to start monitor")
    else:
        print(f"\n✗ Incomplete: got {len(raw)} bytes, need {4 + (length or '?')}")

except Exception as e:
    print(f"\n✗ FAILED: {e}")

Connecting to 192.168.0.99:5066...
  Connected OK
  Sending channel byte 0...
  Reading response...
  Got 7300 bytes (total 7300)
  Payload length from header: 248298 bytes
  Got 14600 bytes (total 21900)
  Got 8760 bytes (total 30660)
  Got 14600 bytes (total 45260)
  Got 13140 bytes (total 58400)
  Got 4380 bytes (total 62780)
  Got 14600 bytes (total 77380)
  Got 5840 bytes (total 83220)
  Got 1460 bytes (total 84680)
  Got 1460 bytes (total 86140)
  Got 1460 bytes (total 87600)
  Got 1460 bytes (total 89060)
  Got 1460 bytes (total 90520)
  Got 14600 bytes (total 105120)
  Got 14600 bytes (total 119720)
  Got 14600 bytes (total 134320)
  Got 14600 bytes (total 148920)
  Got 14600 bytes (total 163520)
  Got 14600 bytes (total 178120)
  Got 14600 bytes (total 192720)
  Got 14600 bytes (total 207320)
  Got 10220 bytes (total 217540)
  Got 1460 bytes (total 219000)
  Got 1460 bytes (total 220460)
  Got 1460 bytes (total 221920)
  Got 1460 bytes (total 223380)
  Got 1460 bytes (total 22

## Cell 8 — Push cavity settings

In [10]:
CAV_RANGE     = [[0.15, 0.50], [1.70, 2.00]]
CAV_LOCKPOINT = 1.80
CAV_PID       = {"P": 0.0, "I": 2.0, "D": 0.0, "I_val": 0, "limit": [-0.99, 0.99]}

Lock.update_setting("Cav", "Master", "range",     CAV_RANGE)
Lock.update_setting("Cav", "Master", "lockpoint", CAV_LOCKPOINT)
Lock.update_setting("Cav", "Master", "enabled",   True)
Lock.update_setting("Cav", "Master", "PID",       CAV_PID)
print("Settings pushed")

check if lockpoint is still fine
Settings pushed


## Cell 9 — Start monitor (only run after Cell 7 shows SUCCESS)

In [11]:
# Only run this after Cell 7 confirms port 5066 works!
Lock.start_monitor("Cav")
print("Monitor started. Qt window should be visible.")
print("To stop: Lock.stop_monitor('Cav')")

Setting up monitor on main thread (Qt window)...
Settings added to plot
Monitor started (scan_mon mode — port 5066)
Monitor started. Qt window should be visible.
To stop: Lock.stop_monitor('Cav')


## Cell 10 — Stop everything

In [12]:
Lock.stop_monitor("Cav")
import time; time.sleep(0.5)
Lock.stop_loop("Cav")
import time; time.sleep(0.5)
Lock.close()
print("All stopped.")

Caught exception: [WinError 10061] No connection could be made because the target machine actively refused it
send: could not connect to ('192.168.0.99', 5000) — board unreachable?
All stopped.
